# Infra-Bench — SatlasPretrain S2 Full Fine-Tune (spatial split, 3 seeds)

Full fine-tuning evaluation of SatlasPretrain S2 on Infra-Bench, complementing
the linear-probe result reported in `infra_fm_satlaspretrain_eval_v2_spatial.ipynb`.
Motivated by coauthor feedback (Konrad Wessels) and GEO-Bench conventions:
full fine-tuning is the headline evaluation for EO FMs, not just linear
probing.

## What's different vs. the LP notebook

| Aspect | Linear probe v2 | Fine-tune v1 |
|---|---|---|
| Backbone | Frozen (`p.requires_grad = False`) | Fully trainable (`freeze=False`) |
| Optimizer param scope | Head only (~13k params) | Head + full backbone (~87M) |
| Learning rate | 1e-3 (head only) | **1e-4** (all params) per Bastani et al. 2023 |
| Weight decay | 1e-4 | 1e-4 |
| LR schedule | None | **Cosine annealing + 500-step linear warmup**, per-batch stepping |
| Epochs | 25 | 25 (unchanged for comparability) |
| Batch size | 16 | 16 (unchanged) |
| Class weight cap | 10.0 | 10.0 (unchanged) |
| Spatial split artifact | `asset_id_to_split_v1.parquet` | same (direct LP comparability) |
| BEST_CKPT_BEFORE_TEST | present | preserved |
| Per-epoch `metrics.jsonl` append | present | preserved (adds `current_lr`) |

## First-conv adapter under fine-tuning (S2-specific)

The SatlasS2 backbone's first Conv2d was pretrained on 9-channel input
(Satlas's 9-band S2). We use 7 channels (bands 0..6). At init the LP
notebook builds a new 7-ch first conv by averaging the pretrained 9-ch
weights and broadcasting the mean to each new channel — a channel-agnostic
adapter that lets LP evaluate the pretrained representation with our
band selection but loses per-band specialization from pretraining.

Under **full fine-tune**, this first conv is trainable like every other
param, and gradient descent can re-specialize the 7 channels toward
band-specific filters. This is a genuine advantage of fine-tune vs LP
for the S2 backbone and worth citing in Methods as an FT-specific
representational recovery mechanism.

## What's unchanged

- Dataset pipeline (`NpyS2Dataset`, `MultiSectorLabelWrapper`, `SubsetView`)
- Spatial split artifact and old-vs-new-split diagnostic
- Data normalization (percentile 2nd/98th → [0, 1])
- Band selection ([0..6], 7 channels)
- Per-sector F1 v2 definition (`_per_sector_v2`)
- Evaluation function and confusion-matrix machinery
- 3 training seeds: 314, 271, 161

## Additional aggregate JSON fields (fine-tune only)

- `finetune_protocol`: dict recording the exact hyperparameters used
- `peak_gpu_gb` per seed: `torch.cuda.max_memory_allocated()` during training
- `wall_time_s` per seed: total wall clock of `train_one_seed`

## Outputs (only after `SMOKE_ONLY=False`)

- `/.../results/fm_eval_satlas_s2_finetune_v1/satlas_s2_finetune_v1_seed{314,271,161}_results.json`
- `/.../results/fm_eval_satlas_s2_finetune_v1/satlas_s2_finetune_v1_aggregate.json`
- `/.../results/fm_eval_satlas_s2_finetune_v1/confusion_matrix_satlas_s2_finetune_v1_aggregate.png`
- Per-seed subdirs with `checkpoint_best.pt`, `checkpoint_final.pt`, `metrics.jsonl`

## Smoke test

Default `SMOKE_ONLY=True` runs 2 epochs on seed 314 and reports:
- Trainable-parameter count (should be ~87M, not ~13k)
- Peak GPU memory after one train step
- Per-epoch LR trajectory (with scheduler built for full 25 epochs, previews real schedule)
- Divergence check on train_loss
- Extrapolated total wall clock for the full 3-seed run

Flip to `SMOKE_ONLY = False` in the smoke cell to launch the full run.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Runtime -> Change runtime type -> GPU before training.')


Mounted at /content/drive
GPU available: True
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB


In [2]:
%%capture
!pip install -q satlaspretrain-models scikit-learn pyarrow


In [3]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# Only load_split_artifact is needed at training time. If the curation
# zip predates Phase 1, fall back to an inline definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import (zip is pre-Phase-1): {e}. Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


Extracting /content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip -> /content/infrabench_repo ...
done.
Could not import (zip is pre-Phase-1): No module named 'curation.utils.spatial_blocking'. Using inline fallback.


In [4]:
import os
from pathlib import Path

DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL      = '/content/datasets'
OUTPUT_DIR          = f'{DRIVE_ROOT}/results/fm_eval_satlas_s2_finetune_v1'
SPLIT_ARTIFACT_PATH = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':                  'water.water_works',   # legacy manifest tag
    'water.water_works':                      'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

# S2-specific: bands 0..6 from our .npy storage (7 channels, no reorder).
S2_BAND_INDICES = [0, 1, 2, 3, 4, 5, 6]

# S2 normalization: percentile (2nd/98th) → [0, 1]. Same as LP notebook.
PERC_LO = 2.0
PERC_HI = 98.0

# ---- Fine-tune hyperparameters ----
# Bastani et al. 2023 (SatlasPretrain ICCV): AdamW, lr=1e-4, cosine annealing
# with 500-step linear warmup, weight_decay=1e-4. Same recipe as SatlasS1.
IMAGE_SIZE       = 224
FT_EPOCHS        = 25
FT_BATCH         = 16
FT_LR            = 1e-4
FT_WD            = 1e-4
FT_WARMUP_STEPS  = 500
WEIGHT_CAP       = 10.0
SEEDS            = [314, 271, 161]

# Resume-from-checkpoint (mirrors CROMA FT): detect existing checkpoint_final.pt
# and pick up from the recorded epoch. Set False to disable.
RESUME_FROM_CHECKPOINT = True

# Auto-skip smoke check on resume: when any full-run seed dir already contains
# checkpoint_final.pt, treat it as a reconnect after disconnect and bypass the
# 2-epoch smoke. Set False to force smoke regardless.
AUTO_SKIP_SMOKE_IF_RESUMING = True

RUN_NAME_PREFIX  = 'satlas_s2_finetune_v1'
CONFUSION_CMAP   = 'Blues'
CONFUSION_TITLE_PREFIX = 'SatlasS2 fine-tune v1'

FINETUNE_PROTOCOL = {
    'backbone_lr':      FT_LR,
    'head_lr':          FT_LR,
    'weight_decay':     FT_WD,
    'scheduler':        'cosine_with_linear_warmup',
    'warmup_steps':     FT_WARMUP_STEPS,
    'llrd':             None,
    'autocast':         False,
    'grad_checkpointing': False,
    'grad_accum_steps': 1,
    'epochs':           FT_EPOCHS,
    'batch_size':       FT_BATCH,
    'effective_batch_size': FT_BATCH,
    'class_weight_cap': WEIGHT_CAP,
    'seeds':            list(SEEDS),
    'unfreeze_scope':   'all_backbone_params_including_9to7_adapter',
    'reference':        'Bastani et al. 2023 (SatlasPretrain, ICCV)',
}


def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f'Output dir:            {OUTPUT_DIR}')
print(f'Split artifact:        {SPLIT_ARTIFACT_PATH}')
print(f'S2 band indices:       {S2_BAND_INDICES}  (7 channels from bands 0..6)')
print(f'S2 normalization:      percentile [{PERC_LO}, {PERC_HI}] -> [0, 1]')
print(f'Training seeds:        {SEEDS}')
print(f'Fine-tune protocol:    {FT_EPOCHS} epochs, batch {FT_BATCH}, lr {FT_LR}, '
      f'wd {FT_WD}, cosine + {FT_WARMUP_STEPS}-step warmup')
print(f'Class weight cap:      {WEIGHT_CAP}')


Output dir:            /content/drive/MyDrive/infra_fm/results/fm_eval_satlas_s2_finetune_v1
Split artifact:        /content/drive/MyDrive/infra_fm/data/spatial_split/asset_id_to_split_v1.parquet
S2 band indices:       [0, 1, 2, 3, 4, 5, 6]  (7 channels from bands 0..6)
S2 normalization:      percentile [2.0, 98.0] -> [0, 1]
Training seeds:        [314, 271, 161]
Fine-tune protocol:    25 epochs, batch 16, lr 0.0001, wd 0.0001, cosine + 500-step warmup
Class weight cap:      10.0


In [5]:
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')


def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m: continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS: continue
        key = (region, sector)
        if key in seen: continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir
    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            return None
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [COPY]   {region:<22s} {sector:<10s} ({n} tiles)')
        return target_dir
    t0 = time.time()
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
ready = []
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local and (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))
print(f'\nReady: {len(ready)} (region, sector) pairs')


  [EXTRACT]africa                 energy     4s (949 tiles)
  [EXTRACT]africa                 telecom    2s (1 tiles)
  [EXTRACT]africa                 transport  5s (1000 tiles)
  [EXTRACT]africa                 water      5s (892 tiles)
  [EXTRACT]asia                   energy     4s (889 tiles)
  [EXTRACT]asia                   telecom    3s (28 tiles)
  [EXTRACT]asia                   transport  6s (891 tiles)
  [EXTRACT]asia                   water      6s (958 tiles)
  [EXTRACT]australia-oceania      energy     4s (1002 tiles)
  [EXTRACT]australia-oceania      telecom    3s (12 tiles)
  [EXTRACT]australia-oceania      transport  4s (1000 tiles)
  [EXTRACT]australia-oceania      water      5s (1000 tiles)
  [EXTRACT]central-america        energy     6s (999 tiles)
  [EXTRACT]central-america        telecom    2s (1 tiles)
  [EXTRACT]central-america        transport  5s (566 tiles)
  [EXTRACT]central-america        water      5s (999 tiles)
  [EXTRACT]europe                 energy  

In [6]:
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


def percentile_normalize(arr, lo=PERC_LO, hi=PERC_HI):
    img = arr.astype(np.float32)
    low = np.percentile(img, lo); high = np.percentile(img, hi)
    if high <= low:
        return np.clip(img / 255.0, 0.0, 1.0)
    return np.clip((img - low) / (high - low), 0.0, 1.0)


class NpyS2Dataset(Dataset):
    """Self-contained loader for a single `dataset_<region>_<sector>_v1_1k/`
    folder. Selects the 7 S2 bands [0..6] from our 9-band .npy storage and
    applies percentile_normalize (2-98 percentile -> [0, 1]).

    Mirrors the v1 SatlasS2 path's behavior but inlines the dataset class
    so the notebook doesn't depend on the curation-zip-extracted
    NpyInfrastructureDataset."""
    def __init__(self, dataset_root,
                 band_indices=S2_BAND_INDICES,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.band_indices = list(band_indices)
        self.allowed = set(allowed_asset_types)
        self.max_required_band = max(self.band_indices)

        manifest_path = self.dataset_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'missing manifest.json: {manifest_path}')
        with manifest_path.open() as f:
            manifest = json.load(f)

        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'
        records, dropped = [], Counter()
        for r in records_in:
            at = r.get('asset_type')
            if not at:
                dropped['no_label'] += 1; continue
            if at not in self.allowed:
                dropped['filtered_type'] += 1; continue
            img_file = r.get('image_file')
            if not img_file:
                dropped['no_image_file'] += 1; continue
            p = images_dir / img_file
            if not p.exists():
                dropped['missing_npy'] += 1; continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < self.max_required_band + 1:
                    dropped['too_few_bands'] += 1; continue
            except Exception:
                dropped['load_error'] += 1; continue
            records.append(_Record(path=p, asset_id=str(r.get('asset_id', p.stem)),
                                   asset_type=at))
        if dropped:
            print(f'  NpyS2Dataset({self.dataset_root.name}): dropped '
                  f'{sum(dropped.values())} records ({dict(dropped)})')
        if not records:
            raise RuntimeError(f'no usable records in {self.dataset_root}')
        self.records = records

    def __len__(self): return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)
        arr = arr[self.band_indices, :, :]              # (7, H, W) S2 only
        arr = percentile_normalize(arr)                  # -> [0, 1]
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)
        return {'image': torch.from_numpy(img),
                'asset_id': r.asset_id, 'asset_type': r.asset_type}


class MultiSectorLabelWrapper(Dataset):
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base; self.region = region; self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels, self.asset_ids = [], [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None: continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])
            self.asset_ids.append(r.asset_id)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']
        img = F.interpolate(img.unsqueeze(0),
                            size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False).squeeze(0)
        return {'image': img, 'label': self.labels[idx],
                'asset_id': sample['asset_id'],
                'region': self.region, 'sector': self.sector}


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base; self.indices = indices
        self.region = base.region; self.sector = base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


source_datasets = {}
for region, sector, local in ready:
    base = NpyS2Dataset(local)
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds): source_datasets[(region, sector)] = ds
print(f'Built {len(source_datasets)} cell datasets')


Built 28 cell datasets


In [7]:
# Load the spatial split artifact and slice each cell into train/val/test
# SubsetViews accordingly.
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  splits distribution: {Counter(asset_to_split.values())}')

splits = {}
n_unmapped = 0
for key, ds in source_datasets.items():
    region, sector = key
    tr_idx, va_idx, te_idx = [], [], []
    for i in range(len(ds)):
        asset_id = ds.asset_ids[i]
        sp = asset_to_split.get(asset_id)
        if sp == 'train':   tr_idx.append(i)
        elif sp == 'val':   va_idx.append(i)
        elif sp == 'test':  te_idx.append(i)
        else:                n_unmapped += 1
    splits[key] = {
        'train': SubsetView(ds, tr_idx),
        'val':   SubsetView(ds, va_idx),
        'test':  SubsetView(ds, te_idx),
    }

if n_unmapped > 0:
    print(f'NOTE: {n_unmapped} tiles in source_datasets have no split assignment '
          '(likely AlphaEarth-missing; will be excluded from train/val/test).')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])
print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')
print(f'  total in splits: {len(train_global) + len(val_global) + len(test_global)}')


Loaded split artifact: 18,750 asset_id -> split entries
  splits distribution: Counter({'train': 13087, 'val': 2851, 'test': 2812})

Global: train=13087  val=2856  test=2813
  total in splits: 18756


In [8]:
# Diagnostic: regenerate the v1 random stratified split inside the
# notebook for an exact-match comparison. Same logic as v1's
# stratified_split (per-(region, sector) cell, by-class, seed=42).
print('=' * 76)
print('Diagnostic: train/val/test transition table (old random -> new spatial)')
print('=' * 76)


def old_stratified_split(dataset_labels, train_frac=0.7, val_frac=0.15, seed=42):
    by_class = {}
    for i, label in enumerate(dataset_labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


old_split = {}
for key in sorted(source_datasets):
    ds = source_datasets[key]
    tr, va, te = old_stratified_split(ds.labels)
    for i in tr: old_split[ds.asset_ids[i]] = 'train'
    for i in va: old_split[ds.asset_ids[i]] = 'val'
    for i in te: old_split[ds.asset_ids[i]] = 'test'

common_ids = set(old_split) & set(asset_to_split)
print(f'Comparing on {len(common_ids):,} tiles in both old and new splits')
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], asset_to_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')
print('\n(Expected ~47% changed — matches CROMA v2 and AlphaEarth v2; the '
      'spatial split is the same; only the training data layer differs across FMs.)')


Diagnostic: train/val/test transition table (old random -> new spatial)
Comparing on 18,750 tiles in both old and new splits

old \ new      train       val       test
--------------------------------------------------
train           9120      1983       1978
val             1946       410        415
test            2021       458        419

Unchanged: 9,949 (53.1%)
Changed:   8,801 (46.9%)

(Expected ~47% changed — matches CROMA v2 and AlphaEarth v2; the spatial split is the same; only the training data layer differs across FMs.)


In [9]:
import torch.nn as nn
import satlaspretrain_models as spm


class SatlasS2Backbone(nn.Module):
    """SatlasPretrain Sentinel-2 backbone, with 9->7 Conv2d adapter
    preserved EXACTLY from v1.

    Notes on the adapter (preserved from v1, known limitation):
      The pretrained Sentinel2_SwinB_SI_MS first conv has in_channels=9
      (Satlas's pretraining used 9 S2 bands). We feed 7 channels. The
      adapter averages the 9 pretrained input-channel weights together
      and broadcasts the mean to all 7 new input channels — so the new
      first conv is channel-agnostic, losing band-specific filters from
      pretraining. This is a known limitation acknowledged in v1; v2
      preserves it for direct comparability with v1 numbers."""
    EXPECTED_BANDS_IN = 9
    NAME = 'SatlasPretrain_Sentinel2_SwinB_SI_MS'
    FEATURE_DIM = 1024

    def __init__(self, in_channels=7, freeze=True):
        super().__init__()
        weights_manager = spm.Weights()
        self.backbone = weights_manager.get_pretrained_model(
            model_identifier='Sentinel2_SwinB_SI_MS', fpn=False,
        )
        self._adapt_first_conv(in_channels)
        self.feature_dim = self.FEATURE_DIM
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def _adapt_first_conv(self, in_channels):
        for name, module in self.backbone.named_modules():
            if isinstance(module, nn.Conv2d) and module.in_channels == self.EXPECTED_BANDS_IN:
                old = module
                new = nn.Conv2d(in_channels, old.out_channels,
                                kernel_size=old.kernel_size, stride=old.stride,
                                padding=old.padding, bias=(old.bias is not None))
                with torch.no_grad():
                    avg = old.weight.mean(dim=1, keepdim=True)
                    new.weight.copy_(avg.expand(-1, in_channels, -1, -1))
                    if old.bias is not None:
                        new.bias.copy_(old.bias)
                if '.' in name:
                    parent_path, attr = name.rsplit('.', 1)
                    parent = self.backbone
                    for p in parent_path.split('.'):
                        parent = getattr(parent, p)
                    setattr(parent, attr, new)
                else:
                    setattr(self.backbone, name, new)
                print(f'  Adapted first conv: {self.EXPECTED_BANDS_IN} -> {in_channels} channels at "{name}"')
                return
        raise RuntimeError(f'No Conv2d with in_channels={self.EXPECTED_BANDS_IN} found.')

    def forward(self, x):
        feature_maps = self.backbone(x)
        last = feature_maps[-1]
        return last.mean(dim=[2, 3])


class InfraBenchClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(backbone.feature_dim, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


# Backbone factory used by train_one_seed.
def BACKBONE_FACTORY(freeze=True):
    return SatlasS2Backbone(in_channels=7, freeze=freeze)


print('SatlasS2Backbone + InfraBenchClassifier defined.')


SatlasS2Backbone + InfraBenchClassifier defined.


In [10]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=WEIGHT_CAP):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """Per-sector F1 v2 — matches LP notebook exactly."""
    cm = np.array(cm)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition, matches LP).'
        )
    return result


def train_one_seed(seed, *, train_set, val_set, test_set,
                   num_epochs=None, scheduler_total_epochs=None,
                   allow_resume=None,
                   run_name_override=None):
    """Full fine-tune training loop. Structurally identical to S1's
    train_one_seed. See S1 notebook for scheduler_total_epochs rationale."""
    if num_epochs is None:
        num_epochs = FT_EPOCHS
    if scheduler_total_epochs is None:
        scheduler_total_epochs = num_epochs
    if allow_resume is None:
        allow_resume = RESUME_FROM_CHECKPOINT
    if allow_resume is None:
        allow_resume = RESUME_FROM_CHECKPOINT
    set_seed(seed)
    print(f'\n--- seed {seed}  fine-tune ---')
    print(f'    training loop: {num_epochs} epochs   scheduler length: {scheduler_total_epochs} epochs')

    backbone = BACKBONE_FACTORY(freeze=False)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Total params:     {total_params:>13,}')
    print(f'  Trainable params: {trainable_params:>13,}')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=FT_BATCH, shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True,
                              generator=g)
    val_loader   = DataLoader(val_set,   batch_size=FT_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=FT_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW(model.parameters(), lr=FT_LR, weight_decay=FT_WD)

    steps_per_epoch = len(train_loader)
    total_steps     = scheduler_total_epochs * steps_per_epoch
    warmup_iters    = min(FT_WARMUP_STEPS, total_steps - 1)
    warmup_sched    = LinearLR(optimizer, start_factor=1e-3, end_factor=1.0,
                                total_iters=warmup_iters)
    cosine_sched    = CosineAnnealingLR(optimizer, T_max=max(total_steps - warmup_iters, 1))
    scheduler       = SequentialLR(optimizer,
                                   schedulers=[warmup_sched, cosine_sched],
                                   milestones=[warmup_iters])

    run_name = run_name_override or f'{RUN_NAME_PREFIX}_seed{seed}_finetune'
    ckpt_dir = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt     = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt    = ckpt_dir / 'checkpoint_final.pt'
    metrics_jsonl = ckpt_dir / 'metrics.jsonl'
    # Resume-from-checkpoint (mirrors CROMA FT): if a prior session wrote
    # checkpoint_final.pt for this seed, load its state and continue from the
    # recorded epoch. Otherwise fresh start + truncate metrics.jsonl.
    history, best_val_f1, best_epoch = [], -1.0, -1
    start_epoch = 0
    resumed_epochs = 0
    if allow_resume and final_ckpt.exists():
        ckpt = torch.load(final_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            try:
                scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            except NameError:
                pass  # notebook has no scheduler in scope
        if 'scaler_state_dict' in ckpt:
            try:
                scaler.load_state_dict(ckpt['scaler_state_dict'])
            except NameError:
                pass  # notebook has no GradScaler in scope
        history = ckpt.get('history', [])
        best_val_f1 = ckpt.get('best_val_f1', -1.0)
        best_epoch = ckpt.get('best_epoch', -1)
        start_epoch = int(ckpt.get('epoch', 0))
        resumed_epochs = start_epoch
        print(f'  [RESUME] Loaded {final_ckpt.name}: '
              f'{start_epoch}/{num_epochs} epochs done, '
              f'best_val_f1={best_val_f1:.4f} at epoch {best_epoch}')
    else:
        metrics_jsonl.write_text('', encoding='utf-8')
    t_seed_start = time.time()

    for epoch in range(start_epoch, num_epochs):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        current_lr = optimizer.param_groups[0]['lr']
        entry = {
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'current_lr': current_lr,
            'time_s': time.time() - t0,
        }
        history.append(entry)
        with metrics_jsonl.open('a', encoding='utf-8') as f:
            f.write(json.dumps(entry) + '\n'); f.flush(); os.fsync(f.fileno())

        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch = epoch + 1
            marker = ' *'
            torch.save({'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'val_macro_f1': val['macro_f1'],
                        'history': history}, best_ckpt)
        # Persist per-epoch checkpoint_final.pt for resume-safety on disconnect.
        _fckpt = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'best_val_f1': best_val_f1,
            'best_epoch': best_epoch,
        }
        try: _fckpt['scheduler_state_dict'] = scheduler.state_dict()
        except NameError: pass
        try: _fckpt['scaler_state_dict'] = scaler.state_dict()
        except NameError: pass
        torch.save(_fckpt, final_ckpt)
        # Persist per-epoch checkpoint_final.pt for resume-safety on disconnect.
        _fckpt = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'best_val_f1': best_val_f1,
            'best_epoch': best_epoch,
        }
        try: _fckpt['scheduler_state_dict'] = scheduler.state_dict()
        except NameError: pass
        try: _fckpt['scaler_state_dict'] = scaler.state_dict()
        except NameError: pass
        torch.save(_fckpt, final_ckpt)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  lr={current_lr:.2e}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    torch.save({'epoch': num_epochs,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history, 'best_val_f1': best_val_f1,
                'best_epoch': best_epoch}, final_ckpt)

    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'  [BEST_CKPT_BEFORE_TEST] restored epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_macro_f1"]:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)

    wall_time_s = time.time() - t_seed_start
    peak_gpu_gb = (torch.cuda.max_memory_allocated(DEVICE) / 1e9
                   if torch.cuda.is_available() else 0.0)

    return {
        'run_name':      run_name,
        'backbone':      backbone.NAME,
        'condition':     'full_finetune',
        'num_epochs':    num_epochs,
        'scheduler_total_epochs': scheduler_total_epochs,
        'seed':          seed,
        'best_val_f1':   best_val_f1,
        'best_epoch':    best_epoch,
        'tail_mean_f1':  float(np.mean(tail)),
        'tail_std_f1':   float(np.std(tail)),
        'history':       history,
        'resumed_epochs': int(resumed_epochs),
        'test':          test,
        'tested_with':   tested_with,
        'peak_gpu_gb':   float(peak_gpu_gb),
        'wall_time_s':   float(wall_time_s),
        'total_params':      int(total_params),
        'trainable_params':  int(trainable_params),
    }


print('Fine-tune training infrastructure ready.')
print('  - AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)')
print('  - Cosine annealing + 500-step linear warmup, per-batch stepping')
print('  - scheduler_total_epochs decoupled from num_epochs')
print('  - Tracks peak_gpu_gb + wall_time_s per seed')


Device: cuda
Fine-tune training infrastructure ready.
  - AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
  - Cosine annealing + 500-step linear warmup, per-batch stepping
  - scheduler_total_epochs decoupled from num_epochs
  - Tracks peak_gpu_gb + wall_time_s per seed


In [11]:
# ============================================================================
# SMOKE CHECK — 2 epochs on seed 314. See S1 notebook for the schedule-preview
# rationale (scheduler_total_epochs=FT_EPOCHS so LR trajectory matches the
# first 2 epochs of the full 25-epoch run).
# ============================================================================
SMOKE_ONLY   = False
SMOKE_EPOCHS = 2
SMOKE_SEED   = SEEDS[0]


# ---- Auto-skip smoke on resume ---------------------------------------------
# If any full-run seed dir already contains checkpoint_final.pt, this is a
# reconnect after a disconnect — running smoke would waste ~10 min on a
# distinct SMOKE_ ckpt dir before the multi-seed cell can pick up the
# interrupted run. Bypass smoke and go straight to resume.
_full_run_ckpts_present = any(
    (Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{s}_finetune' / 'checkpoint_final.pt').exists()
    for s in SEEDS
)
if AUTO_SKIP_SMOKE_IF_RESUMING and _full_run_ckpts_present:
    print('=' * 76)
    print('AUTO_SKIP_SMOKE: existing full-run checkpoint_final.pt detected in one')
    print(f'or more of SEEDS={SEEDS} ckpt dirs. Interpreting this as a reconnect')
    print('after a disconnect — skipping the 2-epoch smoke check.')
    print()
    print('SMOKE_ONLY set to False; the multi-seed cell below will resume.')
    print('=' * 76)
    SMOKE_ONLY = False
    smoke_result = None
else:
    print(f'SMOKE_ONLY = {SMOKE_ONLY}')
    print(f'Smoke check: {SMOKE_EPOCHS} epochs on seed {SMOKE_SEED}, '
          f'lr={FT_LR}, wd={FT_WD}, cosine+{FT_WARMUP_STEPS}-step warmup')
    print(f'Scheduler built for FULL {FT_EPOCHS}-epoch protocol (previews real LR trajectory).')
    print('=' * 76)

    print('\n[1] Instantiate fine-tune model, verify forward pass on one batch...')
    set_seed(SMOKE_SEED)
    sb = SatlasS2Backbone(freeze=False)   # <-- FT delta: all params trainable
    sm = InfraBenchClassifier(sb, num_classes=len(CLASS_NAMES)).to(DEVICE)
    _total  = sum(p.numel() for p in sm.parameters())
    _train  = sum(p.numel() for p in sm.parameters() if p.requires_grad)
    print(f'  Total params:     {_total:>13,}')
    print(f'  Trainable params: {_train:>13,}   (expect ~87M for full S2 fine-tune)')
    assert _train > 80_000_000, (
        f'Trainable param count {_train:,} looks too small for full fine-tune; '
        f'expected ~87M. Backbone freeze may not have been disabled.'
    )

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)

    smoke_loader = DataLoader(train_global, batch_size=FT_BATCH, shuffle=True,
                              num_workers=0, collate_fn=collate)
    batch = next(iter(smoke_loader))
    img = batch['image'].to(DEVICE)
    lbl = batch['label'].to(DEVICE)
    weights = compute_class_weights(train_global).to(DEVICE)
    crit  = nn.CrossEntropyLoss(weight=weights)
    opt   = AdamW(sm.parameters(), lr=FT_LR, weight_decay=FT_WD)

    sm.train()
    opt.zero_grad()
    loss = crit(sm(img), lbl)
    loss.backward()
    opt.step()
    print(f'  One-step loss: {loss.item():.4f}  (finite: {torch.isfinite(loss).item()})')
    assert torch.isfinite(loss).item(), 'Loss is NaN/Inf on first step — abort.'

    if torch.cuda.is_available():
        peak_after_step = torch.cuda.max_memory_allocated(DEVICE) / 1e9
        print(f'  Peak GPU after 1 step: {peak_after_step:.2f} GB')
        if peak_after_step > 22.0:
            print(f'  !! WARNING: {peak_after_step:.2f} GB is close to L4 24 GB limit.')
        del sm, sb, opt

    print(f'\n[2] Running {SMOKE_EPOCHS}-epoch smoke on seed {SMOKE_SEED} '
          f'(schedule built for {FT_EPOCHS} epochs)...')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)

    smoke_result = train_one_seed(
        SMOKE_SEED,
        train_set=train_global,
        val_set=val_global,
        test_set=test_global,
        num_epochs=SMOKE_EPOCHS,
        scheduler_total_epochs=FT_EPOCHS,
            run_name_override=f'{RUN_NAME_PREFIX}_SMOKE_seed{SMOKE_SEED}_finetune',
)

    hist = smoke_result['history']
    train_losses = [h['train_loss'] for h in hist]
    val_f1s      = [h['val_macro_f1'] for h in hist]
    per_epoch_s  = [h['time_s'] for h in hist]
    per_epoch_lr = [h['current_lr'] for h in hist]

    print('\n' + '=' * 76)
    print('SMOKE SUMMARY')
    print('=' * 76)
    print(f'  train_loss trajectory:   {[f"{l:.4f}" for l in train_losses]}')
    print(f'  val_macro_f1 trajectory: {[f"{f:.4f}" for f in val_f1s]}')
    print(f'  end-of-epoch LR:         {[f"{lr:.2e}" for lr in per_epoch_lr]}')
    print(f'  time per epoch:          {[f"{t:.0f}s" for t in per_epoch_s]}')
    print(f'  peak GPU during train:   {smoke_result["peak_gpu_gb"]:.2f} GB')
    print(f'  wall time (smoke seed):  {smoke_result["wall_time_s"]:.0f} s')

    if per_epoch_lr[-1] < FT_LR * 0.5:
        print()
        print('!!' + '=' * 74)
        print(f'!! LR SANITY: end-of-epoch-{SMOKE_EPOCHS} LR = {per_epoch_lr[-1]:.2e}')
        print(f'!! Expected close to FT_LR={FT_LR:.2e} with 25-epoch schedule.')
        print('!!' + '=' * 74)

    avg_epoch_s = float(np.mean(per_epoch_s))
    extrapolated_per_seed_s = avg_epoch_s * FT_EPOCHS
    extrapolated_total_s    = extrapolated_per_seed_s * len(SEEDS)
    def _fmt_hms(s):
        s = int(round(s)); h, r = divmod(s, 3600); m, s = divmod(r, 60)
        return f'{h}h {m}m {s}s' if h else (f'{m}m {s}s' if m else f'{s}s')
    print(f'\n  Extrapolated per-seed:   {_fmt_hms(extrapolated_per_seed_s)}  '
          f'({FT_EPOCHS} epochs × {avg_epoch_s:.0f}s/epoch)')
    print(f'  Extrapolated total:      {_fmt_hms(extrapolated_total_s)}  '
          f'({len(SEEDS)} seeds × {_fmt_hms(extrapolated_per_seed_s)})')

    if len(train_losses) >= 2 and train_losses[-1] >= train_losses[0]:
        print()
        print('!!' + '=' * 74)
        print('!! WARNING: train_loss did NOT decrease over the smoke run.')
        print(f'!!   epoch 1 loss: {train_losses[0]:.4f}')
        print(f'!!   epoch {len(train_losses)} loss: {train_losses[-1]:.4f}')
        print('!!' + '=' * 74)
    else:
        if len(train_losses) >= 2:
            print(f'\n  Train loss decreased by {train_losses[0] - train_losses[-1]:.4f} '
                  f'over smoke. lr={FT_LR} looks stable.')
        print('  Ready to flip SMOKE_ONLY=False for the full 3-seed run.')

    smoke_out = Path(OUTPUT_DIR) / 'smoke_check_results.json'
    with smoke_out.open('w') as f:
        smoke_result_compact = dict(smoke_result)
        json.dump(smoke_result_compact, f, indent=2)
    print(f'\nSmoke results written: {smoke_out}')
    print(f'\nSMOKE_ONLY = {SMOKE_ONLY} — multi-seed cell below will '
          f'{"NOT train" if SMOKE_ONLY else "run all 3 seeds"}.')


SMOKE_ONLY = False
Smoke check: 2 epochs on seed 314, lr=0.0001, wd=0.0001, cosine+500-step warmup
Scheduler built for FULL 25-epoch protocol (previews real LR trajectory).

[1] Instantiate fine-tune model, verify forward pass on one batch...
  Adapted first conv: 9 -> 7 channels at "backbone.backbone.features.0.0"
  Total params:        87,952,365
  Trainable params:    87,952,365   (expect ~87M for full S2 fine-tune)
  One-step loss: 3.6186  (finite: True)
  Peak GPU after 1 step: 5.78 GB

[2] Running 2-epoch smoke on seed 314 (schedule built for 25 epochs)...

--- seed 314  fine-tune ---
    training loop: 2 epochs   scheduler length: 25 epochs
  Adapted first conv: 9 -> 7 channels at "backbone.backbone.features.0.0"
  Total params:        87,952,365
  Trainable params:    87,952,365
  ep   1  loss=2.2278  lr=9.99e-05  val_acc=0.3985  val_f1=0.2658 *
  ep   2  loss=1.7222  lr=9.92e-05  val_acc=0.4391  val_f1=0.3270 *
  [BEST_CKPT_BEFORE_TEST] restored epoch 2 (val_f1=0.3270) before 

In [12]:
# ============================================================================
# Multi-seed fine-tune. Same structure as S1 fine-tune multi-seed.
# ============================================================================
import json as _json
import numpy as np

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping the 3-seed fine-tune run.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        out_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
        if out_path.exists():
            # Skip training: load existing per-seed JSON into per_seed_results
            # so the aggregate step below still works.
            # If a prior run was contaminated (e.g. by the water_works
            # ASSET_TYPE_MAP bug), delete the file on Drive to force a fresh train.
            with out_path.open() as f:
                per_seed_results[seed] = _json.load(f)['finetune']
            print(f'  [SKIP] seed {seed}: existing per-seed JSON at '
                  f'{out_path.name} (delete to force rerun)')
            continue
        result = train_one_seed(seed,
                                train_set=train_global,
                                val_set=val_global,
                                test_set=test_global)
        per_seed_results[seed] = result
        out_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
        with open(out_path, 'w') as f:
            _json.dump({'finetune': result}, f, indent=2)
        print(f'\n  saved {out_path}')

    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {'mean': float(arr.mean()), 'std': float(arr.std(ddof=0)),
                'per_seed': [float(v) for v in arr]}

    agg = {}
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {'class': CLASS_NAMES[i], 'idx': i,
         'mean_f1':  float(per_class_arr[:, i].mean()),
         'std_f1':   float(per_class_arr[:, i].std(ddof=0)),
         'per_seed': [float(v) for v in per_class_arr[:, i]]}
        for i in range(len(CLASS_NAMES))
    ]

    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1'] for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.mean(f1s)),
            'std_macro_f1':  float(np.std(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
               for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.nanmean(f1s)),
            'std_macro_f1':  float(np.nanstd(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    agg['seeds']                       = SEEDS
    agg['split_artifact']              = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note']  = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']

    agg['finetune_protocol'] = FINETUNE_PROTOCOL
    agg['peak_gpu_gb'] = {
        'mean':     float(np.mean([per_seed_results[s]['peak_gpu_gb'] for s in SEEDS])),
        'per_seed': {int(s): float(per_seed_results[s]['peak_gpu_gb']) for s in SEEDS},
    }
    agg['wall_time_s'] = {
        'mean':     float(np.mean([per_seed_results[s]['wall_time_s'] for s in SEEDS])),
        'total':    float(np.sum([per_seed_results[s]['wall_time_s'] for s in SEEDS])),
        'per_seed': {int(s): float(per_seed_results[s]['wall_time_s']) for s in SEEDS},
    }

    # Guard: skip aggregate JSON + confusion PNG writes when SEEDS isn't the
    # full 3-seed protocol. Prevents a single-seed rerun (e.g. running just
    # seed 161) from clobbering an eventual 3-seed aggregate. Per-seed JSONs
    # still get saved — filenames are seed-indexed, no collision risk.
    FULL_PROTOCOL_SEEDS = [314, 271, 161]
    _is_full_run = set(SEEDS) == set(FULL_PROTOCOL_SEEDS)
    if _is_full_run:
        agg_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_aggregate.json'
        with open(agg_path, 'w') as f:
            _json.dump(agg, f, indent=2)
        print(f'\nAggregate saved: {agg_path}')
    else:
        print(f'\nNOTE: SEEDS = {SEEDS}, not the full protocol {FULL_PROTOCOL_SEEDS}.')
        print('      Skipping aggregate JSON + confusion PNG writes to preserve')
        print('      any existing 3-seed aggregate. Per-seed JSON(s) still saved.')
        print('      Once all 3 seeds complete, run the standalone aggregator')
        print('      cell to build the 3-seed aggregate from the per-seed JSONs.')

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    cm_norm = cm_sum.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm), where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap=CONFUSION_CMAP, vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    mf1 = agg['test_macro_f1']
    ax.set_title(f'{CONFUSION_TITLE_PREFIX} — aggregate confusion (summed over '
                 f'{len(SEEDS)} seeds, row-normalized)\n'
                 f'macro F1 = {mf1["mean"]:.3f} +/- {mf1["std"]:.3f}, '
                 f'seeds = {SEEDS}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    if _is_full_run:
        cm_path = Path(OUTPUT_DIR) / f'confusion_matrix_{RUN_NAME_PREFIX}_aggregate.png'
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Confusion matrix saved: {cm_path}')
    else:
        plt.close()
        print('Skipping aggregate confusion matrix PNG (single-seed rerun).')

    def _fmt_hms(s):
        s = int(round(s)); h, r = divmod(s, 3600); m, s = divmod(r, 60)
        return f'{h}h {m}m {s}s' if h else (f'{m}m {s}s' if m else f'{s}s')
    print('\n' + '=' * 76)
    print(f'{CONFUSION_TITLE_PREFIX} — aggregate ({len(SEEDS)} seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f} +/- {agg["test_macro_f1"]["std"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f} +/- {agg["test_accuracy"]["std"]:.4f}')
    print(f'Peak GPU:      {agg["peak_gpu_gb"]["mean"]:.2f} GB (mean across seeds)')
    print(f'Wall clock:    per-seed {_fmt_hms(agg["wall_time_s"]["mean"])}, '
          f'total {_fmt_hms(agg["wall_time_s"]["total"])}')
    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')
    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')



--- seed 314  fine-tune ---
    training loop: 25 epochs   scheduler length: 25 epochs
  Adapted first conv: 9 -> 7 channels at "backbone.backbone.features.0.0"
  Total params:        87,952,365
  Trainable params:    87,952,365
  ep   1  loss=2.2278  lr=9.99e-05  val_acc=0.3985  val_f1=0.2658 *
  ep   2  loss=1.7222  lr=9.92e-05  val_acc=0.4391  val_f1=0.3270 *
  ep   3  loss=1.4641  lr=9.77e-05  val_acc=0.5147  val_f1=0.3663 *
  ep   4  loss=1.2714  lr=9.53e-05  val_acc=0.5357  val_f1=0.3909 *
  ep   5  loss=1.1301  lr=9.22e-05  val_acc=0.4741  val_f1=0.3766
  ep   6  loss=0.9787  lr=8.84e-05  val_acc=0.5182  val_f1=0.3932 *
  ep   7  loss=0.8275  lr=8.40e-05  val_acc=0.5049  val_f1=0.3991 *
  ep   8  loss=0.6863  lr=7.90e-05  val_acc=0.5214  val_f1=0.4207 *
  ep   9  loss=0.5678  lr=7.35e-05  val_acc=0.5175  val_f1=0.4225 *
  ep  10  loss=0.4794  lr=6.77e-05  val_acc=0.5347  val_f1=0.4069
  ep  11  loss=0.3618  lr=6.15e-05  val_acc=0.5599  val_f1=0.4293 *
  ep  12  loss=0.2859  lr=

In [ ]:
"""Aggregate S2 fine-tune across all 3 per-seed JSONs on disk.
Produces the 3-seed aggregate JSON + confusion PNG without retraining.
Run this ONCE after all 3 seeds have completed and their per-seed JSONs
exist under OUTPUT_DIR."""
import json as _json
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT = Path(OUTPUT_DIR)
FULL_SEEDS = [314, 271, 161]

# ---- Load per-seed JSONs from disk ----
per_seed_results = {}
for seed in FULL_SEEDS:
    p = OUT / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
    if not p.exists():
        raise FileNotFoundError(f'missing per-seed JSON: {p}')
    with p.open() as f:
        # S2 FT per-seed JSONs are wrapped as {'finetune': result}
        per_seed_results[seed] = _json.load(f)['finetune']
    print(f'  loaded {p.name}')

def _agg(values):
    arr = np.array(values, dtype=np.float64)
    return {'mean': float(arr.mean()), 'std': float(arr.std(ddof=0)),
            'per_seed': [float(v) for v in arr]}

agg = {}
agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in FULL_SEEDS])
agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc']      for s in FULL_SEEDS])

per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in FULL_SEEDS])
agg['per_class_f1'] = [
    {'class': CLASS_NAMES[i], 'idx': i,
     'mean_f1':  float(per_class_arr[:, i].mean()),
     'std_f1':   float(per_class_arr[:, i].std(ddof=0)),
     'per_seed': [float(v) for v in per_class_arr[:, i]]}
    for i in range(len(CLASS_NAMES))
]

agg['per_sector_f1'] = {}
for sector in SECTOR_TO_CLASS_IDX:
    f1s = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1'] for s in FULL_SEEDS]
    n_seed = per_seed_results[FULL_SEEDS[0]]['test']['per_sector'][sector]['n']
    agg['per_sector_f1'][sector] = {
        'n': n_seed,
        'mean_macro_f1': float(np.mean(f1s)),
        'std_macro_f1':  float(np.std(f1s, ddof=0)),
        'per_seed':      [float(v) for v in f1s],
    }

region_keys = sorted({r for s in FULL_SEEDS for r in per_seed_results[s]['test']['per_region']})
agg['per_region_f1'] = {}
for region in region_keys:
    f1s = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
           for s in FULL_SEEDS]
    n_seed = per_seed_results[FULL_SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
    agg['per_region_f1'][region] = {
        'n': n_seed,
        'mean_macro_f1': float(np.nanmean(f1s)),
        'std_macro_f1':  float(np.nanstd(f1s, ddof=0)),
        'per_seed':      [float(v) for v in f1s],
    }

agg['seeds']                       = FULL_SEEDS
agg['split_artifact']              = SPLIT_ARTIFACT_PATH
agg['per_sector_correction_note']  = per_seed_results[FULL_SEEDS[0]]['test']['per_sector_correction_note']
agg['finetune_protocol']           = FINETUNE_PROTOCOL
agg['peak_gpu_gb'] = {
    'mean':     float(np.mean([per_seed_results[s]['peak_gpu_gb'] for s in FULL_SEEDS])),
    'per_seed': {int(s): float(per_seed_results[s]['peak_gpu_gb']) for s in FULL_SEEDS},
}
agg['wall_time_s'] = {
    'mean':     float(np.mean([per_seed_results[s]['wall_time_s'] for s in FULL_SEEDS])),
    'total':    float(np.sum([per_seed_results[s]['wall_time_s'] for s in FULL_SEEDS])),
    'per_seed': {int(s): float(per_seed_results[s]['wall_time_s']) for s in FULL_SEEDS},
}

agg_path = OUT / f'{RUN_NAME_PREFIX}_aggregate.json'
with agg_path.open('w') as f:
    _json.dump(agg, f, indent=2)
print(f'\nAggregate saved: {agg_path}')


  loaded satlas_s2_finetune_v1_seed314_results.json
  loaded satlas_s2_finetune_v1_seed271_results.json
  loaded satlas_s2_finetune_v1_seed161_results.json

Aggregate saved: /content/drive/MyDrive/infra_fm/results/fm_eval_satlas_s2_finetune_v1/satlas_s2_finetune_v1_aggregate.json
